In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q19/annotated/checkpoints/pre_cell_2.pickle

trying: ['load_lineitem']
me:  1
trying: ['load_part']
me:  2
trying: ['pd']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['DATA_ROOT']
me:  0
trying: ['lineitem']


me:  1
trying: ['part']
me:  2
trying: ['tpch_parent']
me:  0


Declaring variable pd
Declaring variable STORAGE_OPTS
Declaring variable DATA_ROOT
Declaring variable tpch_parent
Declaring variable load_lineitem
Declaring variable lineitem
Declaring variable load_part
Declaring variable part


In [4]:
%%RecordEvent
%%time
### cell 2 ###

# Define container and ship mode lists using string literals
SM_SMALL = ["SM BOX", "SM CASE", "SM PACK", "SM PKG"]
MED_CONTAINER = ["MED BAG", "MED BOX", "MED PACK", "MED PKG"]
LG_CONTAINER = ["LG BOX", "LG CASE", "LG PACK", "LG PKG"]
SHIPMODES = ["AIR", "AIRREG"]

# Filter lineitem with vectorized between and isin
li_qty = (
    lineitem.L_QUANTITY.between(4, 14)
    | lineitem.L_QUANTITY.between(15, 25)
    | lineitem.L_QUANTITY.between(26, 36)
)
li_ship = (
    (lineitem.L_SHIPINSTRUCT == "DELIVER IN PERSON")
    & lineitem.L_SHIPMODE.isin(SHIPMODES)
)
flineitem = lineitem[li_qty & li_ship]

# Filter part using isin and between
small_cond = (
    (part.P_BRAND == "Brand#31")
    & part.P_CONTAINER.isin(SM_SMALL)
    & part.P_SIZE.between(1, 5)
)
med_cond = (
    (part.P_BRAND == "Brand#43")
    & part.P_CONTAINER.isin(MED_CONTAINER)
    & part.P_SIZE.between(1, 10)
)
lg_cond = (
    (part.P_BRAND == "Brand#43")
    & part.P_CONTAINER.isin(LG_CONTAINER)
    & part.P_SIZE.between(1, 15)
)
fpart = part[small_cond | med_cond | lg_cond]

# Join and apply final filtering
jn = flineitem.merge(fpart, left_on="L_PARTKEY", right_on="P_PARTKEY")

cond1 = (
    (jn.P_BRAND == "Brand#31")
    & jn.P_CONTAINER.isin(SM_SMALL)
    & jn.L_QUANTITY.between(4, 14)
    & (jn.P_SIZE <= 5)
)
cond2 = (
    (jn.P_BRAND == "Brand#43")
    & jn.P_CONTAINER.isin(MED_CONTAINER)
    & jn.L_QUANTITY.between(15, 25)
    & (jn.P_SIZE <= 10)
)
cond3 = (
    (jn.P_BRAND == "Brand#43")
    & jn.P_CONTAINER.isin(LG_CONTAINER)
    & jn.L_QUANTITY.between(26, 36)
    & (jn.P_SIZE <= 15)
)
j = jn[cond1 | cond2 | cond3]

# Compute total
# Note: using cudf, this will execute on GPU
total = (j.L_EXTENDEDPRICE * (1.0 - j.L_DISCOUNT)).sum()

CPU times: user 103 ms, sys: 23.6 ms, total: 127 ms
Wall time: 127 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q19/rewritten/o4_mini_high/checkpoints/post_cell_2_try_1.pickle

migration speed (bps): 794304405.9586781
---------------------------
variables to migrate:
STORAGE_OPTS 64
cond2 48
MED_CONTAINER 88
cond1 48
li_qty 48
flineitem 48
load_part 144
li_ship 48
pd 72
fpart 48
part 48
total 32
SHIPMODES 72
cond3 48
lg_cond 48
med_cond 48
load_lineitem 144
jn 48
lineitem 48
small_cond 48
tpch_parent 54
SM_SMALL 88
LG_CONTAINER 88
j 48
DATA_ROOT 80
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q19/rewritten/o4_mini_high/checkpoints/post_cell_2_try_1.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['lineitem']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables ['lineitem']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['small_cond', 'li_qty', 'j', 'SHIPMODES', 'flineitem', 'LG_CONTAINER', 'MED_CONTAINER', 'cond2', 'cond3', 'fpart', 'jn', 'lg_cond', 'total', 'SM_SMALL', 'li_ship', 'cond1', 'med_cond']
Future variables []
Modified dataframes
Saved cell execution info to opt_cell_exec_info


In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q19/opt_cell_exec_info_2_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[2], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q19/annotated/checkpoints/post_cell_2.pickle
assert total_opt == total

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
